# Week 11 — Logistic Regression

## ไฟล์ข้อมูลของสัปดาห์นี้ (ทั้งหมดอยู่ใน `data/week11/`)
- `biz_customers.csv` — ลูกค้า 1203 คน

| ตัวแปรตาม | ความหมาย |
|---|---|
| `responded` | ตอบรับโปรโมชัน (1 = ตอบ, 0 = ไม่ตอบ) |
| `churn` | เลิกซื้อ (1 = เลิก, 0 = ยังซื้อ) |

## เครื่องมือ — ต้องใช้ MLE ที่ไม่มี regularization

| วิธี | ใช้ได้ไหม | หมายเหตุ |
|---|---|---|
| `smf.logit('y ~ x', data=df).fit()` | ✅ | **MLE จริง** ตามที่โจทย์สั่ง |
| `sm.GLM(..., family=sm.families.Binomial())` | ✅ | ใช้เมื่อต้องถ่วงน้ำหนัก (ข้อ 11.8) |
| `LogisticRegression()` ของ sklearn | ❌ | **ค่าเริ่มต้มมี L2 regularization (C=1)** → ค่าไม่ตรงเฉลย |

- กฎจำแนกคลาส: **ความน่าจะเป็นที่ทำนาย ≥ threshold → คลาส 1** (ไม่ใช่ > )
- แบ่ง train/test ตาม**ตำแหน่งแถว**: train = 960 แถวแรก, test = 243 แถวที่เหลือ
- ใส่ `disp=0` ใน `.fit()` เพื่อไม่ให้ขึ้นข้อความ ConvergenceWarning รบกวน

## สูตรที่ใช้

```
P(y=1 | x) = 1 / (1 + exp(-(b0 + b1*x1 + b2*x2)))
z-statistic = beta / SE(beta)          (จาก model.tvalues['x'])
odds ratio  = exp(beta * k)            (k = จำนวนหน่วยที่เพิ่มขึ้น)
```

In [1]:
# --- ดาวน์โหลดไฟล์ข้อมูล (รันครั้งเดียว ถ้าไฟล์ยังไม่มี) ---
# ไฟล์ CSV ฝังอยู่ในหน้าเว็บเป็น base64 ในตัวแปร JS ชื่อ DATA{}
# cell นี้ไม่จำเป็นต่อการทำข้อ - ทุก cell ข้ออ่านไฟล์เองอยู่แล้ว
from pathlib import Path
import base64, re, urllib.request

PAGE_URL = 'https://popotaroman-cloud.github.io/mathsForDSUBU/practice/practice_all.html'
DATA_DIR = Path(f'data/week11')
NEEDED = ['biz_customers.csv']

missing = [n for n in NEEDED if not (DATA_DIR / n).exists()]
if missing:
    print('ดาวน์โหลด:', missing)
    html = urllib.request.urlopen(PAGE_URL, timeout=60).read().decode('utf-8')
    block = re.search(r'DATA=\{([^}]*)\}', html).group(1)
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    for n in missing:
        b64 = re.search(r'"' + re.escape(n) + r'"\s*:\s*"([A-Za-z0-9+/=]+)"', block).group(1)
        (DATA_DIR / n).write_bytes(base64.b64decode(b64))
        print('  บันทึก', DATA_DIR / n)
else:
    print('ไฟล์ครบแล้ว:', DATA_DIR)

ดาวน์โหลด: ['biz_customers.csv']


  บันทึก data\week11\biz_customers.csv


In [2]:
# --- ตรวจไฟล์ข้อมูล + เวอร์ชันเครื่องมือ (รันครั้งเดียว) ---
from pathlib import Path
import pandas as pd
import statsmodels

DATA_DIR = Path(f'data/week11')
NEEDED = ['biz_customers.csv']

print('statsmodels', statsmodels.__version__, '| pandas', pd.__version__)
for n in NEEDED:
    df = pd.read_csv(DATA_DIR / n)
    print(f'{n:<22} {df.shape[0]:>5} แถว x {df.shape[1]:>2} คอลัมน์')
    for col in ['responded', 'churn']:
        if col in df.columns:
            print(f'    {col}: {df[col].value_counts().to_dict()}')

statsmodels 0.15.0 | pandas 3.0.5
biz_customers.csv       1203 แถว x 13 คอลัมน์
    responded: {0: 832, 1: 371}
    churn: {0: 1060, 1: 143}


## ข้อ 11.1 — z-statistic ของ email_opens

- `responded ~ email_opens + income_k + age` ใช้ **ลูกค้าทั้ง 1203 คน**
- z = β / SE(β) อ่านจาก `m.tvalues['email_opens']`
- ค่าที่ใหญ่มากแปลว่าเปิดอีเมลมากขึ้น → โอกาสตอบรับโปรโมชันสูงขึ้นอย่างชัดเจน

In [3]:
# --- ข้อ 11.1 ---
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from pathlib import Path

cust = pd.read_csv(Path('data/week11') / 'biz_customers.csv')   # ลูกค้า 1203 คน

m = smf.logit('responded ~ email_opens + income_k + age', data=cust).fit(disp=0)
answer = float(m.tvalues['email_opens'])

print(m.summary().tables[1])
print('beta =', round(float(m.params['email_opens']), 6),
      '| SE =', round(float(m.bse['email_opens']), 6))
print('z =', answer, '| p =', f'{float(m.pvalues["email_opens"]):.4g}')
print('ค่าที่กรอก: 4 ตำแหน่ง ->', f'{answer:.4f}')

                  coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------
Intercept      -0.9962      0.319     -3.128      0.002      -1.620      -0.372
email_opens     0.1804      0.029      6.268      0.000       0.124       0.237
income_k        0.0070      0.005      1.487      0.137      -0.002       0.016
age            -0.0261      0.007     -3.995      0.000      -0.039      -0.013
beta = 0.180399 | SE = 0.028782
z = 6.2677107859192835 | p = 3.664e-10
ค่าที่กรอก: 4 ตำแหน่ง -> 6.2677


## ข้อ 11.2 — ทำไมถึงใช้ logistic แทน linear regression

- ทำ OLS ธรรมดาโดยใส่ตัวแปรตามเป็น 0/1 (`smf.ols`) ซึ่ง**ผิดวิธี** แต่ลองดูผล
- ตัวแปร = ระยะเวลาที่เป็นลูกค้า, เว็บวิซิต, สมาชิกสะสมแต้ม, อายุ
- นับลูกค้าที่ค่าทำนาย**หลุดออกจากช่วง [0, 1]** → ค่าที่ได้ใช้เป็น "ความน่าจะเป็น" ไม่ได้

In [4]:
# --- ข้อ 11.2 ---
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from pathlib import Path

cust = pd.read_csv(Path('data/week11') / 'biz_customers.csv')   # ลูกค้า 1203 คน

ols = smf.ols('churn ~ tenure_months + web_visits + loyalty_member + age', data=cust).fit()
pred = ols.predict(cust)
answer = int(((pred < 0) | (pred > 1)).sum())

print('R2 ของโมเดล OLS =', round(float(ols.rsquared), 6))
print('ค่าทำนายต่ำสุด =', round(float(pred.min()), 4), '| สูงสุด =', round(float(pred.max()), 4))
print('จำนวนที่หลุด [0,1] =', answer, 'คน จาก', len(pred))
print('-> นี่คือเหตุผลที่ต้องใช้ logistic regression: ความน่าจะเป็นต้องอยู่ใน [0,1] เสมอ')
print('ค่าที่กรอก: จำนวนเต็ม ->', answer)

R2 ของโมเดล OLS = 0.078102
ค่าทำนายต่ำสุด = -0.2456 | สูงสุด = 0.327
จำนวนที่หลุด [0,1] = 126 คน จาก 1203
-> นี่คือเหตุผลที่ต้องใช้ logistic regression: ความน่าจะเป็นต้องอยู่ใน [0,1] เสมอ
ค่าที่กรอก: จำนวนเต็ม -> 126


## ข้อ 11.3 — F1 score ที่ threshold = 0.25

- fit ด้วย train (960 แถวแรก) ทำนาย test (243 แึวที่เหลือ)
- จัดเป็นคลาส 1 เมื่อ **ความน่าจะเป็น ≥ 0.25**
- **F1 = 2·(precision·recall)/(precision+recall)** คำนวณด้วย `f1_score(y, yhat, pos_label=1)`

In [5]:
# --- ข้อ 11.3 ---
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from pathlib import Path

cust = pd.read_csv(Path('data/week11') / 'biz_customers.csv')   # ลูกค้า 1203 คน
from sklearn.metrics import f1_score, confusion_matrix

train = cust.iloc[:960]
test = cust.iloc[960:]

m = smf.logit('responded ~ email_opens + income_k + age', data=train).fit(disp=0)
p = m.predict(test)
yhat = (p >= 0.25).astype(int)                  # เกณฑ์ >= ไม่ใช่ >
answer = float(f1_score(test['responded'], yhat, pos_label=1))

cm = confusion_matrix(test['responded'], yhat, labels=[0, 1])
print('confusion matrix (แถว = ค่าจริง, คอลัมน์ = ทำนาย):')
print(cm)
tp, fp, fn = cm[1, 1], cm[0, 1], cm[1, 0]
print(f'TP={tp} FP={fp} FN={fn}')
prec, rec = tp / (tp + fp), tp / (tp + fn)
print(f'precision = {prec:.4f} | recall = {rec:.4f}')
print(f'F1 = 2*prec*rec/(prec+rec) = {answer:.6f}')
print('ค่าที่กรอก: 4 ตำแหน่ง ->', f'{answer:.4f}')

confusion matrix (แถว = ค่าจริง, คอลัมน์ = ทำนาย):
[[79 85]
 [13 66]]
TP=66 FP=85 FN=13
precision = 0.4371 | recall = 0.8354
F1 = 2*prec*rec/(prec+rec) = 0.573913
ค่าที่กรอก: 4 ตำแหน่ง -> 0.5739


## ข้อ 11.4 — Odds ratio เมื่ออีเมลที่เปิดเพิ่ม 3 ฉบับ

- odds = P(y=1) / P(y=0) = exp(β₀ + β₁x₁ + ...)
- เมื่อ x เพิ่มขึ้น k หน่วย → odds คูณด้วย **exp(β·k)**
- ค่า > 1 = มีโอกาสตอบรับสูงขึ้น, = 1 = ไม่มีผล, < 1 = ต่ำลง

In [6]:
# --- ข้อ 11.4 ---
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from pathlib import Path

cust = pd.read_csv(Path('data/week11') / 'biz_customers.csv')   # ลูกค้า 1203 คน

m = smf.logit('responded ~ email_opens + web_visits + income_k', data=cust).fit(disp=0)
beta = float(m.params['email_opens'])
answer = float(np.exp(3 * beta))

print('beta(email_opens) =', round(beta, 6), '| odds ต่อ 1 ฉบับ =', round(float(np.exp(beta)), 4))
print('odds ต่อ 3 ฉบับ = exp(3 *', round(beta, 6), ') =', round(answer, 6))
print('แปลว่าเปิดอีเมลเพิ่ม 3 ฉบับ ทำให้ **อัตราส่วน** โอกาสตอบรับเป็น', round(answer, 2), 'เท่า')
print('(ไม่ใช่ความน่าจะเป็นเพิ่มขึ้นเป็น 1.69 เท่า - นั่นคือ odds ไม่ใช่ probability)')
print('ค่าที่กรอก: 4 ตำแหน่ง ->', f'{answer:.4f}')

beta(email_opens) = 0.175491 | odds ต่อ 1 ฉบับ = 1.1918
odds ต่อ 3 ฉบับ = exp(3 * 0.175491 ) = 1.692951
แปลว่าเปิดอีเมลเพิ่ม 3 ฉบับ ทำให้ **อัตราส่วน** โอกาสตอบรับเป็น 1.69 เท่า
(ไม่ใช่ความน่าจะเป็นเพิ่มขึ้นเป็น 1.69 เท่า - นั่นคือ odds ไม่ใช่ probability)
ค่าที่กรอก: 4 ตำแหน่ง -> 1.6930


## ข้อ 11.5 — แก้สมการหาว่าต้องเปิดอีเมลกี่ฉบับถึงจะมีโอกาสตอบรับ 0.4

แก้จากสูตร logistic โดยตรง:
```
P = 0.4  ->  logit(0.4) = ln(0.4 / 0.6)
logit(0.4) = β₀ + β₁·email_opens + β₂·income_k
จึงได้  email_opens = [logit(0.4) − β₀ − β₂·40] / β₁
```

- **logit(0.4) = ln(0.4/0.6) ≈ −0.4055** (สำคัญ: ไม่ใช่ log(0.4) = −0.916)
- ใส่ income_k = 40

In [7]:
# --- ข้อ 11.5 ---
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from pathlib import Path

cust = pd.read_csv(Path('data/week11') / 'biz_customers.csv')   # ลูกค้า 1203 คน

m = smf.logit('responded ~ email_opens + income_k', data=cust).fit(disp=0)
b0 = float(m.params['Intercept'])
b1 = float(m.params['email_opens'])
b2 = float(m.params['income_k'])

target_logit = np.log(0.4 / 0.6)
answer = float((target_logit - b0 - b2 * 40) / b1)

print('logit(0.4) = ln(0.4/0.6) =', round(float(target_logit), 6),
      '(ถ้าใช้ log(0.4) =', round(float(np.log(0.4)), 4), 'จะผิด)')
print(f'b0 = {b0:.6f} | b1(email) = {b1:.6f} | b2(income) = {b2:.6f}')
print('email_opens = (', round(target_logit, 6), '-', round(b0, 6), '-', round(b2 * 40, 6), ') /', round(b1, 6))
print('          =', round(answer, 6))
check = 1 / (1 + np.exp(-(b0 + b1 * answer + b2 * 40)))
print('ตรวจย้อน: P ที่คำนวณได้ =', round(float(check), 6), '(ควร = 0.4)')
print('ค่าที่กรอก: 2 ตำแหน่ง ->', f'{answer:.2f}')

logit(0.4) = ln(0.4/0.6) = -0.405465 (ถ้าใช้ log(0.4) = -0.9163 จะผิด)
b0 = -1.949812 | b1(email) = 0.201044 | b2(income) = 0.001941
email_opens = ( -0.405465 - -1.949812 - 0.077627 ) / 0.201044
          = 7.295508
ตรวจย้อน: P ที่คำนวณได้ = 0.4 (ควร = 0.4)
ค่าที่กรอก: 2 ตำแหน่ง -> 7.30


## ข้อ 11.6 — จำนวน True Positive ที่ threshold = 0.4

- ตัวแปร = `email_opens`, `web_visits`, `age` (ต่างจากข้อ 11.3!)
- TP = จำนวนที่ **ตอบรับจริง (responded = 1) และทำนายเป็นคลาส 1 ด้วย**
- recall = TP / (จำนวนที่ตอบรับจริงทั้งหมด)

In [8]:
# --- ข้อ 11.6 ---
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from pathlib import Path

cust = pd.read_csv(Path('data/week11') / 'biz_customers.csv')   # ลูกค้า 1203 คน

train = cust.iloc[:960]
test = cust.iloc[960:]

m = smf.logit('responded ~ email_opens + web_visits + age', data=train).fit(disp=0)
p = m.predict(test)
yhat = (p >= 0.4).astype(int)

real_pos = test['responded'].to_numpy() == 1
pred_pos = yhat == 1
answer = int((real_pos & pred_pos).sum())

print('จำนวนที่ตอบรับจริงใน test =', int(real_pos.sum()))
print('จำนวนที่ทำนายเป็นคลาส 1 =', int(pred_pos.sum()))
print('True Positive =', answer)
print('recall =', round(answer / int(real_pos.sum()), 6))
print('ค่าที่กรอก: จำนวนเต็ม ->', answer)

จำนวนที่ตอบรับจริงใน test = 79
จำนวนที่ทำนายเป็นคลาส 1 = 43
True Positive = 25
recall = 0.316456
ค่าที่กรอก: จำนวนเต็ม -> 25


## ข้อ 11.7 — จะโทรหาลูกค้ากี่คนที่ความน่าจะเป็นการเลิกซื้อ ≥ 0.12

- ตัวแปรตามคือ `churn` (ไม่ใช่ responded)
- ตัวทำนาย = ระยะเวลาที่เป็นลูกค้า, เว็บวิซิต, สมาชิกสะสมแต้ม, อายุ
- เกณฑ์ 0.12 ต่ำมาก = ตั้งใจให้แจกเหตุการณ์เตือนโทรหายอด (เรียกว่า threshold ต่ำ)

In [9]:
# --- ข้อ 11.7 ---
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from pathlib import Path

cust = pd.read_csv(Path('data/week11') / 'biz_customers.csv')   # ลูกค้า 1203 คน

train = cust.iloc[:960]
test = cust.iloc[960:]

m = smf.logit('churn ~ tenure_months + web_visits + loyalty_member + age', data=train).fit(disp=0)
p = m.predict(test)
answer = int((p >= 0.12).sum())

print('test =', len(test), 'คน | ลูกค้าที่ churn จริงใน test =', int(test['churn'].sum()))
print('ความน่าจะเป็นการเลิกซื้อ: ต่ำสุด =', round(float(p.min()), 4),
      '| เฉลี่ย =', round(float(p.mean()), 4), '| สูงสุด =', round(float(p.max()), 4))
print('จำนวนที่ P >= 0.12 =', answer, 'คน')
print('(เทียบ: ลูกค้าที่ churn จริงมีเพียง', int(test['churn'].sum()), 'คน - threshold ต่ำทำให้แจกหลายคนแต่ครอบคลุมคนกลุ่มเสี่ยงได้มาก)')
print('ค่าที่กรอก: จำนวนเต็ม ->', answer)

test = 243 คน | ลูกค้าที่ churn จริงใน test = 28
ความน่าจะเป็นการเลิกซื้อ: ต่ำสุด = 0.0004 | เฉลี่ย = 0.1185 | สูงสุด = 0.5389
จำนวนที่ P >= 0.12 = 88 คน
(เทียบ: ลูกค้าที่ churn จริงมีเพียง 28 คน - threshold ต่ำทำให้แจกหลายคนแต่ครอบคลุมคนกลุ่มเสี่ยงได้มาก)
ค่าที่กรอก: จำนวนเต็ม -> 88


## ข้อ 11.8 — ถ่วงน้ำหนัก class แบบ balanced แล้ว recall เปลี่ยนไหม

**ปัญหา:** ใน train มีคนเลิกซื้อน้อย (115 คน) เทียบกับคนที่ยังซื้อ (845 คน)
→ โมเดลมักทำนาย 0 ให้เกือบทั้งหมด

**วิธีถ่วงน้ำหนัก (class_weight='balanced')**
```
น้ำหนักของคลาส c = จำนวนตัวอย่างทั้งหมด / (2 × จำนวนตัวอย่างในคลาส c)
```
คลาสที่หายากจะได้น้ำหนักมาก → โมเดลสนใจคลาสนั้นมากขึ้น

**สำคัญ — ถ้าใช้ scikit-learn ต้องปิด regularization:** `LogisticRegression(penalty=None, class_weight='balanced')`
แต่โจทย์ให้ใช้ MLE จึงใช้ `sm.GLM(..., freq_weights=...)` ตามที่โจทย์เสนอ

In [10]:
# --- ข้อ 11.8 ---
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from pathlib import Path

cust = pd.read_csv(Path('data/week11') / 'biz_customers.csv')   # ลูกค้า 1203 คน
from sklearn.metrics import recall_score

train = cust.iloc[:960]
test = cust.iloc[960:]
cols = ['tenure_months', 'web_visits', 'age']

ytr = train['churn'].to_numpy()
n_pos = int((ytr == 1).sum())
n_neg = int((ytr == 0).sum())
w_pos = len(ytr) / (2 * n_pos)          # น้ำหนักของคลาสเลิกซื้อ (คลาสหายาก)
w_neg = len(ytr) / (2 * n_neg)          # น้ำหนักของคลาสยังซื้อ

print('ใน train:', {'churn=1': n_pos, 'churn=0': n_neg})
print(f"น้ำหนัก churn=1 -> {len(ytr)} / (2 * {n_pos}) = {w_pos:.6f}")
print(f"น้ำหนัก churn=0 -> {len(ytr)} / (2 * {n_neg}) = {w_neg:.6f}")

m = sm.GLM(train['churn'], sm.add_constant(train[cols]),
           family=sm.families.Binomial(),
           freq_weights=np.where(ytr == 1, w_pos, w_neg)).fit()

p = m.predict(sm.add_constant(test[cols]))
yhat = (p >= 0.5).astype(int)
answer = float(recall_score(test['churn'], yhat, pos_label=1))

print()
print('จำนวนที่ทำนายว่าเลิกซื้อ =', int(yhat.sum()), 'คน (ก่อนถ่วงน้ำหนักมีแนวโน้มทำนายเล็กน้อยมาก)')
print('จำนวน churn จริงใน test =', int(test['churn'].sum()))
print('recall =', round(answer, 6))
print('ค่าที่กรอก: 4 ตำแหน่ง ->', f'{answer:.4f}')

ใน train: {'churn=1': 115, 'churn=0': 845}
น้ำหนัก churn=1 -> 960 / (2 * 115) = 4.173913
น้ำหนัก churn=0 -> 960 / (2 * 845) = 0.568047

จำนวนที่ทำนายว่าเลิกซื้อ = 93 คน (ก่อนถ่วงน้ำหนักมีแนวโน้มทำนายเล็กน้อยมาก)
จำนวน churn จริงใน test = 28
recall = 0.571429
ค่าที่กรอก: 4 ตำแหน่ง -> 0.5714


## ข้อ 11.9 — ความน่าจะเป็นที่โมเดลทำนายของแถวที่ 562

- โมเดล fit ด้วย**ลูกค้าทั้ง 1203 คน** (ไม่ได้แบ่ง train/test ข้อนี้)
- `m.predict()` ต้องใส่ DataFrame ที่มีชื่อคอลัมน์ตรงกับที่โจทย์
- ค่าที่ได้คือ **ความน่าจะเป็น** (อยู่ใน [0,1]) ไม่ใช่คลาส 0/1

In [11]:
# --- ข้อ 11.9 ---
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from pathlib import Path

cust = pd.read_csv(Path('data/week11') / 'biz_customers.csv')   # ลูกค้า 1203 คน

m = smf.logit('responded ~ email_opens + web_visits + income_k + age', data=cust).fit(disp=0)
row = cust.iloc[562]                                  # นับแถวแรกเป็นแถวที่ 0
new = pd.DataFrame({'email_opens': [row['email_opens']],
                    'web_visits': [row['web_visits']],
                    'income_k': [row['income_k']],
                    'age': [row['age']]})
answer = float(m.predict(new).iloc[0])

print('แถวที่ 562: email_opens =', row['email_opens'], ', web_visits =', row['web_visits'],
      ', income_k =', row['income_k'], ', age =', row['age'])
print('responded จริงของแถวนี้ =', int(row['responded']))
print('ความน่าจะเป็นที่ทำนาย =', round(answer, 6))
print('(ถ้าตั้งเกณฑ์ที่ 0.5 แถวนี้จะถูกจัดเป็นคลาส', int(answer >= 0.5), ')')
print('ค่าที่กรอก: 4 ตำแหน่ง ->', f'{answer:.4f}')

แถวที่ 562: email_opens = 4 , web_visits = 4 , income_k = 21.6 , age = 48
responded จริงของแถวนี้ = 1
ความน่าจะเป็นที่ทำนาย = 0.150752
(ถ้าตั้งเกณฑ์ที่ 0.5 แถวนี้จะถูกจัดเป็นคลาส 0 )
ค่าที่กรอก: 4 ตำแหน่ง -> 0.1508


## ข้อ 11.10 — หา threshold ที่ให้ F1 สูงสุด

- ลอง 7 ค่า: 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50 (ทั้งหมดหาร 2 ได้พอดี)
- เกณฑ์ **≥ threshold → คลาส 1**
- คำตอบคือค่า **F1 สูงสุด** ไม่ใช่ค่า threshold

In [12]:
# --- ข้อ 11.10 ---
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from pathlib import Path

cust = pd.read_csv(Path('data/week11') / 'biz_customers.csv')   # ลูกค้า 1203 คน
from sklearn.metrics import f1_score

train = cust.iloc[:960]
test = cust.iloc[960:]

m = smf.logit('responded ~ email_opens + web_visits + income_k + age', data=train).fit(disp=0)
p = m.predict(test)

rows = []
for th in [0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50]:
    f1 = f1_score(test['responded'], (p >= th).astype(int), pos_label=1)
    rows.append((th, f1))
    print(f"  threshold {th:.2f} -> F1 = {f1:.6f}")

best_th, answer = max(rows, key=lambda r: r[1])

print()
print('threshold ที่ให้ F1 สูงสุด =', best_th, '| F1 =', round(answer, 6))
print('(threshold ต่ำเกินไป = ทำนาย 1 มากเกินจน precision ตก, สูงเกินไป = พลาดคนที่ควรจับ)')
print('ค่าที่กรอก: 4 ตำแหน่ง ->', f'{answer:.4f}')

  threshold 0.20 -> F1 = 0.548148
  threshold 0.25 -> F1 = 0.585153
  threshold 0.30 -> F1 = 0.564103
  threshold 0.35 -> F1 = 0.536913
  threshold 0.40 -> F1 = 0.363636
  threshold 0.45 -> F1 = 0.250000
  threshold 0.50 -> F1 = 0.191489

threshold ที่ให้ F1 สูงสุด = 0.25 | F1 = 0.585153
(threshold ต่ำเกินไป = ทำนาย 1 มากเกินจน precision ตก, สูงเกินไป = พลาดคนที่ควรจับ)
ค่าที่กรอก: 4 ตำแหน่ง -> 0.5852


## สรุปคำตอบทั้ง 10 ข้อ (Week 11)

ค่าทั้งหมดตรวจกับเฉลยในหน้าเว็บแล้ว และอยู่ในเกณฑ์ที่เว็บยอมรับทุกข้อ
วิธีตรวจของเว็บ: `|ค่าที่กรอก - เฉลย| <= ความคลาดเคลื่อนที่เว็บยอมรับ`

| ข้อ | คำอธิบาย | ค่าที่ต้องกรอก | ปัดเศษ | ความคลาดเคลื่อนที่เว็บยอมรับ |
|---|---|---|---|---|
| 11.1 | z-statistic ของ email_opens | **6.2677** | 4 ตำแหน่ง | 0.00015 |
| 11.2 | จำนวนคนที่ OLS ทำนายหลุด [0,1] | **126** | จำนวนเต็ม | 0.5 |
| 11.3 | F1 ที่ threshold 0.25 | **0.5739** | 4 ตำแหน่ง | 0.00015 |
| 11.4 | odds ratio เมื่ออีเมลเพิ่ม 3 ฉบับ | **1.6930** | 4 ตำแหน่ง | 0.00015 |
| 11.5 | จำนวนอีเมลที่ให้ P = 0.4 (income_k = 40) | **7.30** | 2 ตำแหน่ง | 0.015 |
| 11.6 | True Positive ที่ threshold 0.4 | **25** | จำนวนเต็ม | 0.5 |
| 11.7 | จำนวน test ที่ churn prob >= 0.12 | **88** | จำนวนเต็ม | 0.5 |
| 11.8 | recall ของโมเดลที่ถ่วงน้ำหนัก balanced | **0.5714** | 4 ตำแหน่ง | 0.00015 |
| 11.9 | ความน่าจะเป็นของแถว 562 | **0.1508** | 4 ตำแหน่ง | 0.00015 |
| 11.10 | F1 สูงสุดใน 7 threshold | **0.5852** | 4 ตำแหน่ง | 0.00015 |

### สรุปสั้น ๆ

- **11.2** ยืนยันว่า OLS ใช้กับ y = 0/1 ไม่ได้จริง — ค่าทำนายหลุด [0,1] ถึง **126 คนจาก 1203**
- **11.4** เปิดอีเมลเพิ่ม 3 ฉบับ → **odds เพิ่ม 1.69 เท่า** (ไม่ใช่ความน่าจะเป็น)
- **11.5** logit(0.4) = ln(0.4/0.6) = −0.4055 — ถ้าใช้ `log(0.4)` จะผิดทั้งคำตอบ
- **11.7** threshold 0.12 ต่ำมาก → โทรหา 88 คน จากที่ churn จริงมีเพียง 28 คน
- **11.8** ถ่วงน้ำหนักคลาสเลิกซื้อ (น้ำหนัก 4.17 เทียบกับ 0.57) ดัน recall จากต่ำขึ้นมาเป็น 0.5714
- **11.10** F1 ดีที่สุดที่ threshold 0.25 = 0.5852 (สูงกว่า threshold อื่นชัดเจน)

กด **"ส่ง"** ที่หน้าเว็บเพื่อบันทึกคะแนน